# Direction K: Sample-similarity graph augmentation

Tạo graph giữa các mẫu dựa trên độ giống nhau của gen phụ, lấy đặc trưng hàng xóm để thử tăng hiệu năng dự đoán.

In [1]:

import os, re, json, math, shutil, subprocess, warnings
from pathlib import Path
from collections import Counter, defaultdict
warnings.filterwarnings('ignore')
import numpy as np
import pandas as pd
from IPython.display import display
from sklearn.model_selection import train_test_split
from sklearn.feature_selection import chi2
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, balanced_accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, average_precision_score
from sklearn.decomposition import TruncatedSVD
import matplotlib.pyplot as plt
try:
    import xgboost as xgb
    HAS_XGB=True
except Exception:
    HAS_XGB=False

REPO_URL='https://github.com/347251369/Antimicrobial-resistance-prediction-in-Salmonella.git'
DRUGS=['AMP','AUG','AXO','CHL','FOX']
N_REPEATS=3
RANDOM_SEEDS=list(range(42,42+N_REPEATS))
K_FEATURES=200
MAX_CANDIDATE_FEATURES=5000
MODEL_NAMES=['LR_balanced','RF_balanced']
if HAS_XGB:
    MODEL_NAMES.append('XGB_weighted')
THRESHOLD_METRIC='f1'
print('HAS_XGB:', HAS_XGB)
print('N_REPEATS:', N_REPEATS)
print('MODEL_NAMES:', MODEL_NAMES)

BASE_DIR=Path('/content/salmonella_direction_K_sample_similarity_graph_no_drive')
REPO_DIR=BASE_DIR/'Antimicrobial-resistance-prediction-in-Salmonella'
EXTRACT_DIR=BASE_DIR/'extracted'
OUTPUT_DIR=BASE_DIR/'outputs'
for d in [BASE_DIR,EXTRACT_DIR,OUTPUT_DIR]: d.mkdir(parents=True, exist_ok=True)
print('BASE_DIR:',BASE_DIR)


HAS_XGB: True
N_REPEATS: 3
MODEL_NAMES: ['LR_balanced', 'RF_balanced', 'XGB_weighted']
BASE_DIR: /content/salmonella_direction_K_sample_similarity_graph_no_drive


In [2]:

def list_files(root, suffixes=None):
    root=Path(root); files=[]
    for p in root.rglob('*'):
        if p.is_file() and (suffixes is None or p.suffix.lower() in suffixes): files.append(p)
    return files

def find_largest_table(root):
    candidates=list_files(root, ['.csv','.tsv','.txt','.xlsx','.xls'])
    if not candidates: raise FileNotFoundError(f'Không tìm thấy bảng trong {root}')
    candidates=sorted(candidates, key=lambda p:p.stat().st_size, reverse=True)
    for p in candidates[:5]: print(' -', p.name, round(p.stat().st_size/1024/1024,2),'MB')
    return candidates[0]

def read_table_flexible(path):
    path=Path(path)
    if path.suffix.lower()=='.csv': return pd.read_csv(path)
    if path.suffix.lower() in ['.tsv','.txt']:
        df=pd.read_csv(path, sep='\t')
        return pd.read_csv(path) if df.shape[1]==1 else df
    if path.suffix.lower() in ['.xlsx','.xls']: return pd.read_excel(path)
    raise ValueError(path)

def make_sample_index(n): return pd.Index([f'sample_{i}' for i in range(n)], name='sample_id')

def coerce_numeric_features(df):
    out=df.copy(); drop=[]
    for c in list(out.columns):
        if out[c].dtype=='object':
            conv=pd.to_numeric(out[c], errors='coerce')
            if conv.notna().mean()>0.95: out[c]=conv.fillna(0)
            else: drop.append(c)
    if drop: out=out.drop(columns=drop)
    out=out.fillna(0)
    for c in out.columns:
        vals=pd.unique(out[c])
        if len(vals)<=3:
            try: out[c]=out[c].astype(np.int8)
            except Exception: pass
    return out

def parse_label_series(y_raw):
    y=y_raw.copy()
    if isinstance(y, pd.DataFrame):
        cand=[c for c in y.columns if any(k in c.lower() for k in ['label','phenotype','result','concl'])]
        y=y[cand[0] if cand else y.columns[-1]]
    y=y.replace({'S':0,'I':0,'R':1,'s':0,'i':0,'r':1,'Susceptible':0,'Intermediate':0,'Resistant':1,'susceptible':0,'intermediate':0,'resistant':1})
    return pd.to_numeric(y, errors='coerce')

def safe_scores(y_true, y_pred, y_prob):
    return {'accuracy':accuracy_score(y_true,y_pred),'balanced_accuracy':balanced_accuracy_score(y_true,y_pred),'precision':precision_score(y_true,y_pred,zero_division=0),'recall':recall_score(y_true,y_pred,zero_division=0),'f1':f1_score(y_true,y_pred,zero_division=0),'auroc':roc_auc_score(y_true,y_prob) if len(np.unique(y_true))>1 else np.nan,'auprc':average_precision_score(y_true,y_prob) if len(np.unique(y_true))>1 else np.nan}

def choose_threshold(y_val, prob_val, metric='f1'):
    best_t,best=-1,-1
    for t in np.linspace(0.05,0.95,91):
        pred=(prob_val>=t).astype(int)
        score=balanced_accuracy_score(y_val,pred) if metric=='balanced_accuracy' else f1_score(y_val,pred,zero_division=0)
        if score>best: best=score; best_t=t
    return float(best_t), float(best)

def make_model(name, y_train, seed=42):
    if name=='LR_balanced': return LogisticRegression(max_iter=5000,class_weight='balanced',solver='lbfgs',random_state=seed)
    if name=='RF_balanced': return RandomForestClassifier(n_estimators=300,min_samples_leaf=2,class_weight='balanced_subsample',random_state=seed,n_jobs=-1)
    if name=='XGB_weighted':
        pos=max(int(y_train.sum()),1); neg=max(int((y_train==0).sum()),1)
        return xgb.XGBClassifier(n_estimators=250,max_depth=4,learning_rate=0.04,subsample=0.85,colsample_bytree=0.85,eval_metric='logloss',scale_pos_weight=neg/pos,random_state=seed,n_jobs=-1)
    raise ValueError(name)

def fit_predict_with_threshold(model, X_train, y_train, X_val, y_val, X_test, y_test):
    model.fit(X_train,y_train)
    pv=model.predict_proba(X_val)[:,1] if hasattr(model,'predict_proba') else model.decision_function(X_val)
    pt=model.predict_proba(X_test)[:,1] if hasattr(model,'predict_proba') else model.decision_function(X_test)
    th,vm=choose_threshold(y_val,pv,THRESHOLD_METRIC)
    pred=(pt>=th).astype(int)
    s=safe_scores(y_test,pred,pt); s['threshold']=th; s['val_threshold_metric']=vm
    return s

def top_variance_candidates(X_train, max_features=5000):
    if X_train.shape[1]<=max_features: return list(X_train.columns)
    return list(X_train.var(axis=0).sort_values(ascending=False).head(max_features).index)

def select_chi2_topk(X_train, y_train, k=200, max_candidates=5000):
    cand=top_variance_candidates(X_train, max_candidates)
    scores,_=chi2(X_train[cand].clip(lower=0), y_train)
    s=pd.Series(scores,index=cand).replace([np.inf,-np.inf],np.nan).fillna(0)
    return list(s.sort_values(ascending=False).head(k).index)

def find_drug_label_file(REPO_DIR, drug):
    d=REPO_DIR/'data'/'csv'/drug; exact=d/f'{drug}_label.csv'
    if exact.exists(): return exact
    cand=list(d.glob('*label*.csv'))
    if cand: return cand[0]
    raise FileNotFoundError(f'Không tìm thấy label csv cho {drug}')

def load_ready_gene_and_label(REPO_DIR, drug):
    d=REPO_DIR/'data'/'csv'/drug
    X=coerce_numeric_features(pd.read_csv(d/'gene.csv'))
    y=parse_label_series(pd.read_csv(find_drug_label_file(REPO_DIR,drug)))
    mask=y.notna(); X=X.loc[mask.values].reset_index(drop=True); y=y.loc[mask].reset_index(drop=True).astype(int)
    idx=make_sample_index(len(y)); X.index=idx; y.index=idx
    return X,y


In [3]:

if not REPO_DIR.exists():
    !git clone --depth 1 https://github.com/347251369/Antimicrobial-resistance-prediction-in-Salmonella.git "{REPO_DIR}"
else:
    print('Repo đã tồn tại:', REPO_DIR)
!apt-get update -qq
!apt-get install -y unrar > /dev/null

acc_dir=EXTRACT_DIR/'accessory_gene'; acc_dir.mkdir(parents=True, exist_ok=True)
rar=REPO_DIR/'results'/'Roary'/'accessory gene existence matrix.rar'
if not any(acc_dir.glob('*')):
    if rar.exists():
        !unrar x -o+ "{rar}" "{acc_dir}/" > /dev/null
    else:
        local=BASE_DIR/'accessory_gene_existence_matrix.rar'
        url='https://github.com/347251369/Antimicrobial-resistance-prediction-in-Salmonella/raw/main/results/Roary/accessory%20gene%20existence%20matrix.rar'
        !wget -q -O "{local}" "{url}"
        !unrar x -o+ "{local}" "{acc_dir}/" > /dev/null

ready_data={}; rows=[]
for drug in DRUGS:
    Xr,yr=load_ready_gene_and_label(REPO_DIR, drug)
    ready_data[drug]={'X':Xr,'y':yr}
    rows.append({'drug':drug,'n_samples':len(yr),'n_resistant':int(yr.sum()),'n_non_resistant':int((yr==0).sum()),'resistant_rate':float(yr.mean())})
stats_df=pd.DataFrame(rows); display(stats_df); stats_df.to_csv(OUTPUT_DIR/'dataset_stats.csv',index=False)

acc_path=find_largest_table(acc_dir)
X_accessory=coerce_numeric_features(read_table_flexible(acc_path))
if X_accessory.shape[0]==len(next(iter(ready_data.values()))['y']): X_accessory.index=make_sample_index(X_accessory.shape[0])
else: raise ValueError('Accessory rows không khớp label rows')
print('X_accessory:', X_accessory.shape)
display(X_accessory.iloc[:3,:5])


Cloning into '/content/salmonella_direction_K_sample_similarity_graph_no_drive/Antimicrobial-resistance-prediction-in-Salmonella'...
remote: Enumerating objects: 79, done.
remote: Counting objects: 100% (79/79), done.
remote: Compressing objects: 100% (53/53), done.
remote: Total 79 (delta 33), reused 54 (delta 25), pack-reused 0 (from 0)
Receiving objects: 100% (79/79), 2.95 MiB | 16.31 MiB/s, done.
Resolving deltas: 100% (33/33), done.
W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)


,drug,n_samples,n_resistant,n_non_resistant,resistant_rate
0,AMP,1167,199,968,0.170523
1,AUG,1167,139,1028,0.119109
2,AXO,1167,71,1096,0.060840
3,CHL,1167,126,1041,0.107969
4,FOX,1167,71,1096,0.060840


 - accessory gene existence matrix.csv 40.59 MB
X_accessory: (1167, 18125)


,ldtD,golT,GXP82_000609,D1K42_06100,astB
sample_id,,,,,
sample_0,1,1,1,1,1
sample_1,1,1,1,1,1
sample_2,1,1,1,1,1


In [4]:

def binary_jaccard_similarity(A,B):
    A=np.asarray(A,dtype=np.float32); B=np.asarray(B,dtype=np.float32)
    inter=A@B.T; union=A.sum(axis=1,keepdims=True)+B.sum(axis=1,keepdims=True).T-inter
    return inter/np.maximum(union,1e-8)

def make_sample_graph_features(X_train_sel, y_train, X_query_sel, is_train_query=False, top_k=15):
    sim=binary_jaccard_similarity(X_query_sel.values, X_train_sel.values)
    ytr=np.asarray(y_train).astype(float)
    if is_train_query: np.fill_diagonal(sim,-1.0)
    feats=[]
    for i in range(sim.shape[0]):
        row=sim[i].copy(); idx=np.argsort(row)[::-1][:top_k]
        sims=row[idx]; labs=ytr[idx]; valid=sims>=0; sims=sims[valid]; labs=labs[valid]
        if len(sims)==0: feats.append([0,0,0,0,0,0,0]); continue
        w=np.maximum(sims,0); wr=float((w*labs).sum()/w.sum()) if w.sum()>0 else float(labs.mean())
        rs=row[ytr==1]; ss=row[ytr==0]
        feats.append([float(labs.mean()),wr,float(np.max(sims)),float(np.mean(sims)),float(np.max(rs)) if len(rs) else 0,float(np.max(ss)) if len(ss) else 0,float((labs==1).sum())])
    cols=['neighbor_resistant_rate','weighted_neighbor_resistant_rate','max_neighbor_similarity','mean_topk_similarity','max_similarity_to_resistant_train','max_similarity_to_sensitive_train','topk_resistant_count']
    return pd.DataFrame(feats,index=X_query_sel.index,columns=cols)

all_rows=[]
def evaluate_drug(drug):
    y_all=ready_data[drug]['y']; X_all=X_accessory.loc[y_all.index]; rows=[]
    for seed in RANDOM_SEEDS:
        X_trainval,X_test,y_trainval,y_test=train_test_split(X_all,y_all,test_size=0.2,random_state=seed,stratify=y_all)
        X_train,X_val,y_train,y_val=train_test_split(X_trainval,y_trainval,test_size=0.25,random_state=seed+1000,stratify=y_trainval)
        selected=select_chi2_topk(X_train,y_train,K_FEATURES,MAX_CANDIDATE_FEATURES)
        Xtr,Xva,Xte=X_train[selected],X_val[selected],X_test[selected]
        Gtr=make_sample_graph_features(Xtr,y_train,Xtr,True,15); Gva=make_sample_graph_features(Xtr,y_train,Xva,False,15); Gte=make_sample_graph_features(Xtr,y_train,Xte,False,15)
        settings={'accessory_200_only':(Xtr,Xva,Xte),'sample_graph_only':(Gtr,Gva,Gte),'accessory_200_plus_sample_graph':(pd.concat([Xtr.reset_index(drop=True),Gtr.reset_index(drop=True)],axis=1),pd.concat([Xva.reset_index(drop=True),Gva.reset_index(drop=True)],axis=1),pd.concat([Xte.reset_index(drop=True),Gte.reset_index(drop=True)],axis=1))}
        for setting,(A_tr,A_va,A_te) in settings.items():
            for model_name in MODEL_NAMES:
                model=make_model(model_name,y_train,seed)
                scores=fit_predict_with_threshold(model,A_tr,y_train.reset_index(drop=True),A_va,y_val.reset_index(drop=True),A_te,y_test.reset_index(drop=True))
                rows.append({'drug':drug,'seed':seed,'setting':setting,'model':model_name,**scores})
    return pd.DataFrame(rows)
for drug in DRUGS:
    print('\n'+'='*80); print('Direction K:',drug); print('='*80)
    df=evaluate_drug(drug); all_rows.append(df)
    display(df.groupby(['setting','model'])[['balanced_accuracy','f1','auprc']].mean().sort_values('f1',ascending=False).head(10))
results_df=pd.concat(all_rows,ignore_index=True); results_df.to_csv(OUTPUT_DIR/'direction_K_all_results.csv',index=False)



Direction K: AMP


balanced_accuracy        f1  \
setting                         model                                       
accessory_200_only              XGB_weighted           0.907345  0.879991   
                                LR_balanced            0.895704  0.867135   
accessory_200_plus_sample_graph LR_balanced            0.900601  0.861750   
                                XGB_weighted           0.888101  0.843778   
                                RF_balanced            0.890421  0.832669   
sample_graph_only               XGB_weighted           0.890421  0.831110   
                                RF_balanced            0.888703  0.822526   
accessory_200_only              RF_balanced            0.873754  0.808165   
sample_graph_only               LR_balanced            0.875344  0.805440   

                                                 auprc  
setting                         model                   
accessory_200_only              XGB_weighted  0.920836  
                                LR_balanced   0.926743  
accessory_200_plus_sample_graph LR_balanced   0.925575  
                                XGB_weighted  0.892166  
                                RF_balanced   0.906174  
sample_graph_only               XGB_weighted  0.869234  
                                RF_balanced   0.883909  
accessory_200_only              RF_balanced   0.913536  
sample_graph_only               LR_balanced   0.877568


Direction K: AUG


balanced_accuracy        f1  \
setting                         model                                       
accessory_200_only              XGB_weighted           0.968909  0.924968   
                                LR_balanced            0.962957  0.918309   
accessory_200_plus_sample_graph LR_balanced            0.957004  0.911599   
sample_graph_only               LR_balanced            0.949434  0.895140   
accessory_200_plus_sample_graph RF_balanced            0.944290  0.894089   
                                XGB_weighted           0.939147  0.892138   
accessory_200_only              RF_balanced            0.937529  0.881652   
sample_graph_only               RF_balanced            0.909385  0.855046   
                                XGB_weighted           0.913719  0.851109   

                                                 auprc  
setting                         model                   
accessory_200_only              XGB_weighted  0.952487  
                                LR_balanced   0.950298  
accessory_200_plus_sample_graph LR_balanced   0.949765  
sample_graph_only               LR_balanced   0.932303  
accessory_200_plus_sample_graph RF_balanced   0.926754  
                                XGB_weighted  0.933990  
accessory_200_only              RF_balanced   0.923410  
sample_graph_only               RF_balanced   0.919116  
                                XGB_weighted  0.904449


Direction K: AXO


balanced_accuracy        f1  \
setting                         model                                       
accessory_200_only              LR_balanced            0.974675  0.951370   
accessory_200_plus_sample_graph LR_balanced            0.962771  0.939055   
accessory_200_only              XGB_weighted           0.950866  0.921456   
accessory_200_plus_sample_graph XGB_weighted           0.927056  0.892232   
                                RF_balanced            0.904762  0.889550   
accessory_200_only              RF_balanced            0.904004  0.880988   
sample_graph_only               RF_balanced            0.925541  0.880596   
                                XGB_weighted           0.923268  0.855928   
                                LR_balanced            0.932143  0.829630   

                                                 auprc  
setting                         model                   
accessory_200_only              LR_balanced   0.992262  
accessory_200_plus_sample_graph LR_balanced   0.984191  
accessory_200_only              XGB_weighted  0.983856  
accessory_200_plus_sample_graph XGB_weighted  0.944084  
                                RF_balanced   0.972118  
accessory_200_only              RF_balanced   0.947827  
sample_graph_only               RF_balanced   0.957615  
                                XGB_weighted  0.915783  
                                LR_balanced   0.921686


Direction K: CHL


balanced_accuracy        f1  \
setting                         model                                       
accessory_200_plus_sample_graph RF_balanced            0.916810  0.887077   
accessory_200_only              LR_balanced            0.921085  0.876882   
                                RF_balanced            0.920287  0.870714   
accessory_200_plus_sample_graph LR_balanced            0.920287  0.870441   
accessory_200_only              XGB_weighted           0.914418  0.868722   
accessory_200_plus_sample_graph XGB_weighted           0.908549  0.866953   
sample_graph_only               RF_balanced            0.894418  0.845257   
                                LR_balanced            0.892823  0.833769   
                                XGB_weighted           0.888836  0.808395   

                                                 auprc  
setting                         model                   
accessory_200_plus_sample_graph RF_balanced   0.872505  
accessory_200_only              LR_balanced   0.883241  
                                RF_balanced   0.880828  
accessory_200_plus_sample_graph LR_balanced   0.877084  
accessory_200_only              XGB_weighted  0.862362  
accessory_200_plus_sample_graph XGB_weighted  0.846447  
sample_graph_only               RF_balanced   0.844940  
                                LR_balanced   0.863822  
                                XGB_weighted  0.851387


Direction K: FOX


balanced_accuracy        f1  \
setting                         model                                       
accessory_200_plus_sample_graph XGB_weighted           0.973918  0.943210   
accessory_200_only              XGB_weighted           0.973160  0.933891   
                                RF_balanced            0.973160  0.933891   
                                LR_balanced            0.962013  0.930183   
accessory_200_plus_sample_graph LR_balanced            0.962013  0.930183   
sample_graph_only               RF_balanced            0.928571  0.919753   
                                XGB_weighted           0.961255  0.918689   
                                LR_balanced            0.958225  0.884433   
accessory_200_plus_sample_graph RF_balanced            0.915152  0.881975   

                                                 auprc  
setting                         model                   
accessory_200_plus_sample_graph XGB_weighted  0.946172  
accessory_200_only              XGB_weighted  0.952137  
                                RF_balanced   0.920781  
                                LR_balanced   0.929576  
accessory_200_plus_sample_graph LR_balanced   0.924788  
sample_graph_only               RF_balanced   0.940325  
                                XGB_weighted  0.923058  
                                LR_balanced   0.933128  
accessory_200_plus_sample_graph RF_balanced   0.930054

In [5]:

summary=results_df.groupby(['drug','setting','model'])[['balanced_accuracy','precision','recall','f1','auroc','auprc']].agg(['mean','std']).reset_index()
summary.columns=['_'.join([str(x) for x in col if str(x)!='']) if isinstance(col,tuple) else col for col in summary.columns]
summary.to_csv(OUTPUT_DIR/'summary.csv',index=False)
best=[]
for drug in DRUGS:
    sub=summary[summary['drug']==drug].sort_values(['f1_mean','balanced_accuracy_mean','auprc_mean'],ascending=False)
    best.append(sub.iloc[0])
best_df=pd.DataFrame(best); best_df.to_csv(OUTPUT_DIR/'best_by_drug.csv',index=False)
display(best_df[['drug','setting','model','balanced_accuracy_mean','f1_mean','auprc_mean']])
lines=['# Auto conclusion','']
for _,r in best_df.iterrows(): lines.append(f"- {r['drug']}: best = {r['setting']} + {r['model']}; F1={r['f1_mean']:.3f}, balanced accuracy={r['balanced_accuracy_mean']:.3f}, AUPRC={r['auprc_mean']:.3f}.")
conclusion='\n'.join(lines); print(conclusion)
with open(OUTPUT_DIR/'AUTO_CONCLUSION.md','w',encoding='utf-8') as f: f.write(conclusion)
zip_path=BASE_DIR/(BASE_DIR.name+'_outputs.zip')
if zip_path.exists(): zip_path.unlink()
shutil.make_archive(str(zip_path).replace('.zip',''),'zip',OUTPUT_DIR)
print('zip:', zip_path)


,drug,setting,model,balanced_accuracy_mean,f1_mean,auprc_mean
2,AMP,accessory_200_only,XGB_weighted,0.907345,0.879991,0.920836
11,AUG,accessory_200_only,XGB_weighted,0.968909,0.924968,0.952487
18,AXO,accessory_200_only,LR_balanced,0.974675,0.951370,0.992262
31,CHL,accessory_200_plus_sample_graph,RF_balanced,0.916810,0.887077,0.872505
41,FOX,accessory_200_plus_sample_graph,XGB_weighted,0.973918,0.943210,0.946172


# Auto conclusion

- AMP: best = accessory_200_only + XGB_weighted; F1=0.880, balanced accuracy=0.907, AUPRC=0.921.
- AUG: best = accessory_200_only + XGB_weighted; F1=0.925, balanced accuracy=0.969, AUPRC=0.952.
- AXO: best = accessory_200_only + LR_balanced; F1=0.951, balanced accuracy=0.975, AUPRC=0.992.
- CHL: best = accessory_200_plus_sample_graph + RF_balanced; F1=0.887, balanced accuracy=0.917, AUPRC=0.873.
- FOX: best = accessory_200_plus_sample_graph + XGB_weighted; F1=0.943, balanced accuracy=0.974, AUPRC=0.946.
zip: /content/salmonella_direction_K_sample_similarity_graph_no_drive/salmonella_direction_K_sample_similarity_graph_no_drive_outputs.zip
